# TFG V4 vs V5 comparison

Este notebook compara las versiones nuevas `TFG_V4.ipynb` y `TFG_V5.ipynb` usando los datos que ahora recopilan sus análisis finales.

Las versiones actuales ya no generan los CSV antiguos `*_param_methods_all_cases.csv`; por eso esta comparación carga directamente las definiciones de ambos notebooks, ejecuta el mismo optimizador reducido `best_first_hybrid_optimizer` para cada caso y construye tablas propias.

Se comparan:

- `P_valid` final optimizado.
- Repeticiones óptimas `r_opt`.
- Ángulos `theta/pi` y `beta/pi`.
- Mejora respecto a la distribución uniforme `K/W`.
- Eficiencia simple `P_valid / r_opt`.
- Qubits estimados por el layout de registros de cada versión.
- Ganador por caso en probabilidad, repeticiones y eficiencia.

Los resultados se guardan en `analysis_v4_v5_comparison/`. Si los notebooks V4/V5 cambian y quieres forzar recalcular, pon `RECOMPUTE_RESULTS = True` en la celda de cálculo.


In [1]:
import csv
import json
import os
import re
import time
from math import prod
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", os.path.join(os.getcwd(), ".matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", os.path.join(os.getcwd(), ".cache"))
os.environ.setdefault("MPLBACKEND", "Agg")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    pass

plt.rcParams.update({
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
})

V4_COLOR = "#3498db"
V5_COLOR = "#f39c12"
BASELINE_COLOR = "0.45"
POS_COLOR = "#2ecc71"
NEG_COLOR = "#e74c3c"
TIE_COLOR = "0.55"

ROOT = Path(".")
if not (ROOT / "TFG_V4.ipynb").exists() and (ROOT / "TFG" / "TFG_V4.ipynb").exists():
    ROOT = ROOT / "TFG"

V4_NOTEBOOK = ROOT / "TFG_V4.ipynb"
V5_NOTEBOOK = ROOT / "TFG_V5.ipynb"
OUT_DIR = ROOT / "analysis_v4_v5_comparison"
OUT_DIR.mkdir(exist_ok=True)

RECOMPUTE_RESULTS = False
EPS = 1e-10


def savefig(fig, stem):
    fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(OUT_DIR / f"{stem}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)


def slug(text):
    return re.sub(r"[^a-zA-Z0-9_]+", "_", str(text)).strip("_").lower()


def print_table(df, title=None, max_rows=30):
    if title:
        print(f"\n{title}")
    if len(df) > max_rows:
        print(df.head(max_rows).to_string(index=False))
        print(f"... ({len(df) - max_rows} more rows)")
    else:
        print(df.to_string(index=False))


In [2]:
# =========================================================
# Load V4/V5 notebook definitions without running their long output cells
# =========================================================

VERSION_SPECS = {
    "V4": {
        "notebook": V4_NOTEBOOK,
        "prefix": "v4",
        "cases_var": "V4_CASE_STUDIES",
        "definition_cells": [2, 3, 4, 9, 10, 12],
        "optimizer_cell": 14,
        "oracle_label": "C(i) cost phase",
    },
    "V5": {
        "notebook": V5_NOTEBOOK,
        "prefix": "v5",
        "cases_var": "V5_CASE_STUDIES",
        "definition_cells": [2, 3, 4, 10, 11, 13],
        "optimizer_cell": 16,
        "oracle_label": "ΔC(i) delta cost phase",
    },
}


def notebook_source_cell(notebook_path, cell_index):
    nb = json.loads(Path(notebook_path).read_text())
    return "".join(nb["cells"][cell_index].get("source", []))


def optimizer_definitions_only(source):
    marker = "\nFINAL_HYBRID_CASES ="
    if marker not in source:
        raise ValueError("Could not find FINAL_HYBRID_CASES marker in optimizer cell.")
    return source.split(marker, 1)[0]


def load_version_namespace(spec):
    ns = {
        "np": np,
        "pd": pd,
        "plt": plt,
        "Path": Path,
        "re": re,
        "time": time,
        "csv": csv,
        "prod": prod,
    }
    for cell_index in spec["definition_cells"]:
        source = notebook_source_cell(spec["notebook"], cell_index)
        exec(compile(source, f"{spec['notebook']}::cell{cell_index}", "exec"), ns)

    optimizer_source = optimizer_definitions_only(
        notebook_source_cell(spec["notebook"], spec["optimizer_cell"])
    )
    exec(compile(optimizer_source, f"{spec['notebook']}::optimizer_defs", "exec"), ns)
    return ns


version_envs = {version: load_version_namespace(spec) for version, spec in VERSION_SPECS.items()}
print("Loaded namespaces:")
for version, ns in version_envs.items():
    print(
        f"  {version}: cases={len(ns[VERSION_SPECS[version]['cases_var']])}, "
        f"theta_points={ns['HYBRID_THETA_POINTS']}, beta_points={ns['HYBRID_BETA_POINTS']}, "
        f"r_max={ns['HYBRID_R_MAX']}"
    )


Final hybrid shared utilities loaded.
Final hybrid shared utilities loaded.
Loaded namespaces:
  V4: cases=10, theta_points=15, beta_points=15, r_max=60
  V5: cases=10, theta_points=15, beta_points=15, r_max=60


In [3]:
# =========================================================
# Validate shared benchmark cases
# =========================================================

def normalize_case_for_comparison(case):
    return {
        "name": case["name"],
        "description": case.get("description", ""),
        "N": list(case["N"]),
        "M": list(case["M"]),
        "occupied_coords": [tuple(c) for c in case["occupied_coords"]],
        "theta": float(case["theta"]),
        "beta": float(case["beta"]),
        "mixer_method": case.get("mixer_method", "local_geometric"),
    }


v4_cases_raw = version_envs["V4"][VERSION_SPECS["V4"]["cases_var"]]
v5_cases_raw = version_envs["V5"][VERSION_SPECS["V5"]["cases_var"]]
v4_cases = [normalize_case_for_comparison(case) for case in v4_cases_raw]
v5_cases = [normalize_case_for_comparison(case) for case in v5_cases_raw]

if v4_cases != v5_cases:
    print("WARNING: V4_CASE_STUDIES and V5_CASE_STUDIES are not identical.")
    for left, right in zip(v4_cases, v5_cases):
        if left != right:
            print("Mismatch:", left["name"], left, right)
else:
    print(f"OK: V4 and V5 define the same {len(v4_cases)} benchmark cases.")

case_order = [case["name"] for case in v4_cases]
case_labels = [name.replace("_", "\n", 2) for name in case_order]
case_meta = pd.DataFrame(v4_cases)
case_meta.to_csv(OUT_DIR / "v4_v5_case_definitions.csv", index=False)
print_table(case_meta[["name", "N", "M", "occupied_coords", "theta", "beta", "mixer_method"]], "Case definitions")


OK: V4 and V5 define the same 10 benchmark cases.

Case definitions
                            name         N         M                                                                  occupied_coords    theta  beta    mixer_method
           01_1d_tiny_single_gap       [6]       [2]                                                               [(0,), (3,), (4,)] 1.570796  0.30 local_geometric
            02_1d_main_reference       [8]       [2]                                                   [(0,), (1,), (2,), (6,), (7,)] 1.047198  0.60 local_geometric
          03_1d_two_free_regions      [10]       [3]                                                   [(0,), (1,), (7,), (8,), (9,)] 1.047198  0.30 local_geometric
          04_1d_clustered_medium      [16]       [3]                                     [(0,), (1,), (5,), (6,), (7,), (13,), (14,)] 1.047198  0.24 local_geometric
     05_1d_long_clustered_blocks      [32]       [4] [(0,), (1,), (2,), (9,), (10,), (11,), (18,), (19,), (

In [4]:
# =========================================================
# Run or load final-hybrid reduced results for V4 and V5
# =========================================================

def estimate_register_layout(ctx, version):
    N_tot = int(prod(ctx["N"]))
    M_tot = int(prod(ctx["M"]))
    IDX = int(ctx["IDX"])
    if version == "V4":
        return {
            "N_tot": N_tot,
            "M_tot": M_tot,
            "IDX": IDX,
            "m_registers": 1,
            "n_anc_bits": 0,
            "n_anc": 0,
            "estimated_total_qubits": N_tot + IDX + M_tot,
        }

    n_anc_bits = int(np.ceil(np.log2(M_tot + 1)))
    n_anc = 2 * n_anc_bits + 1
    return {
        "N_tot": N_tot,
        "M_tot": M_tot,
        "IDX": IDX,
        "m_registers": 2,
        "n_anc_bits": n_anc_bits,
        "n_anc": n_anc,
        "estimated_total_qubits": N_tot + IDX + 2 * M_tot + n_anc,
    }


def context_for_case(version, case):
    spec = VERSION_SPECS[version]
    ns = version_envs[version]
    return ns[f"{spec['prefix']}_case_context"](case)


def run_version_cases(version):
    spec = VERSION_SPECS[version]
    ns = version_envs[version]
    prefix = spec["prefix"]
    cases = ns[spec["cases_var"]]
    optimizer = ns["best_first_hybrid_optimizer"]

    rows = []
    curve_rows = []
    started = time.time()

    for case in cases:
        ctx = context_for_case(version, case)
        t0 = time.time()
        opt = optimizer(ctx)
        selected = opt["selected"]
        theta_opt = float(selected["theta"])
        beta_opt = float(selected["beta"])
        r_opt = int(selected["r"])
        p_valid = float(selected["p_valid"])
        p_curve = np.asarray(selected["p_curve"], dtype=float)
        layout = estimate_register_layout(ctx, version)

        if "delta_costs" in ctx:
            signal_min = float(np.min(ctx["delta_costs"]))
            signal_max = float(np.max(ctx["delta_costs"]))
            signal_nonzero = int(np.count_nonzero(ctx["delta_costs"]))
        else:
            signal_min = float(np.min(ctx["costs"]))
            signal_max = float(np.max(ctx["costs"]))
            signal_nonzero = int(np.count_nonzero(ctx["costs"]))

        rows.append({
            "version": version,
            "case": ctx["name"],
            "oracle": spec["oracle_label"],
            "W": int(ctx["W"]),
            "K": int(len(ctx["valid_indices"])),
            "P_uniform": float(ctx["P_uniform"]),
            "mixer_method": ctx.get("mixer_method", "local_geometric"),
            "theta_default_over_pi": float(ctx["theta_default"] / np.pi),
            "beta_default_over_pi": float(ctx["beta_default"] / np.pi),
            "theta_opt": theta_opt,
            "beta_opt": beta_opt,
            "theta_over_pi": float(theta_opt / np.pi),
            "beta_over_pi": float(beta_opt / np.pi),
            "r_opt": r_opt,
            "r_star": r_opt,
            "P_valid": p_valid,
            "P_max_reduced_search": float(selected.get("P_max", np.nan)),
            "hardware_score": float(selected.get("score", np.nan)),
            "improvement_over_uniform": float(p_valid - ctx["P_uniform"]),
            "relative_improvement_over_uniform": float((p_valid - ctx["P_uniform"]) / ctx["P_uniform"]) if ctx["P_uniform"] > 0 else np.nan,
            "P_valid_per_repetition": float(p_valid / r_opt) if r_opt > 0 else np.nan,
            "optimizer_seconds": float(time.time() - t0),
            "signal_min": signal_min,
            "signal_max": signal_max,
            "signal_nonzero_count": signal_nonzero,
            **layout,
        })

        for r, p_value in enumerate(p_curve, start=1):
            curve_rows.append({
                "version": version,
                "case": ctx["name"],
                "theta_over_pi": float(theta_opt / np.pi),
                "beta_over_pi": float(beta_opt / np.pi),
                "r": int(r),
                "P_valid": float(p_value),
                "is_selected_r": int(r == r_opt),
            })

        print(
            f"{version} {ctx['name']}: P_valid={p_valid:.6f}, "
            f"r={r_opt}, theta/pi={theta_opt/np.pi:.4f}, beta/pi={beta_opt/np.pi:.4f}, "
            f"time={time.time() - t0:.2f}s"
        )

    print(f"{version} total optimizer time: {time.time() - started:.2f}s")
    return pd.DataFrame(rows), pd.DataFrame(curve_rows)


def load_or_compute_version(version):
    result_path = OUT_DIR / f"{version.lower()}_final_hybrid_results.csv"
    curve_path = OUT_DIR / f"{version.lower()}_selected_repetition_curves.csv"
    if result_path.exists() and curve_path.exists() and not RECOMPUTE_RESULTS:
        print(f"Loading cached {version} results from {result_path}")
        return pd.read_csv(result_path), pd.read_csv(curve_path)

    result_df, curve_df = run_version_cases(version)
    result_df.to_csv(result_path, index=False)
    curve_df.to_csv(curve_path, index=False)
    return result_df, curve_df


v4_results, v4_curves = load_or_compute_version("V4")
v5_results, v5_curves = load_or_compute_version("V5")
all_results = pd.concat([v4_results, v5_results], ignore_index=True)
all_curves = pd.concat([v4_curves, v5_curves], ignore_index=True)
all_results.to_csv(OUT_DIR / "v4_v5_all_final_hybrid_results.csv", index=False)
all_curves.to_csv(OUT_DIR / "v4_v5_selected_repetition_curves.csv", index=False)

print_table(all_results[[
    "version", "case", "P_valid", "r_opt", "theta_over_pi", "beta_over_pi",
    "P_uniform", "improvement_over_uniform", "estimated_total_qubits"
]], "Computed / loaded final-hybrid results")


Loading cached V4 results from analysis_v4_v5_comparison/v4_final_hybrid_results.csv
Loading cached V5 results from analysis_v4_v5_comparison/v5_final_hybrid_results.csv

Computed / loaded final-hybrid results
version                             case  P_valid  r_opt  theta_over_pi  beta_over_pi  P_uniform  improvement_over_uniform  estimated_total_qubits
     V4            01_1d_tiny_single_gap 0.763423     29       0.089286      0.053571   0.200000                  0.563423                      11
     V4             02_1d_main_reference 0.927011     32       0.098214      0.116071   0.285714                  0.641297                      13
     V4           03_1d_two_free_regions 0.964373     35       0.026786      0.062500   0.375000                  0.589373                      16
     V4           04_1d_clustered_medium 0.865485     47       0.107143      0.107143   0.285714                  0.579771                      23
     V4      05_1d_long_clustered_blocks 0.922892     2

In [5]:
# =========================================================
# Direct V4-V5 comparison by case
# =========================================================

v4 = v4_results.set_index("case").loc[case_order].reset_index()
v5 = v5_results.set_index("case").loc[case_order].reset_index()

comparison = v4.merge(v5, on="case", suffixes=("_V4", "_V5"))
comparison["delta_P_V5_minus_V4"] = comparison["P_valid_V5"] - comparison["P_valid_V4"]
comparison["delta_r_V5_minus_V4"] = comparison["r_opt_V5"] - comparison["r_opt_V4"]
comparison["delta_qubits_V5_minus_V4"] = comparison["estimated_total_qubits_V5"] - comparison["estimated_total_qubits_V4"]
comparison["delta_efficiency_V5_minus_V4"] = comparison["P_valid_per_repetition_V5"] - comparison["P_valid_per_repetition_V4"]
comparison["P_winner"] = np.where(
    comparison["delta_P_V5_minus_V4"] > EPS, "V5",
    np.where(comparison["delta_P_V5_minus_V4"] < -EPS, "V4", "tie"),
)
comparison["fewer_repetitions"] = np.where(
    comparison["delta_r_V5_minus_V4"] < 0, "V5",
    np.where(comparison["delta_r_V5_minus_V4"] > 0, "V4", "tie"),
)
comparison["efficiency_winner"] = np.where(
    comparison["delta_efficiency_V5_minus_V4"] > EPS, "V5",
    np.where(comparison["delta_efficiency_V5_minus_V4"] < -EPS, "V4", "tie"),
)
comparison["pareto_probability_repetitions"] = np.select(
    [
        (comparison["delta_P_V5_minus_V4"] >= -EPS) & (comparison["delta_r_V5_minus_V4"] <= 0) &
        ((comparison["delta_P_V5_minus_V4"] > EPS) | (comparison["delta_r_V5_minus_V4"] < 0)),
        (comparison["delta_P_V5_minus_V4"] <= EPS) & (comparison["delta_r_V5_minus_V4"] >= 0) &
        ((comparison["delta_P_V5_minus_V4"] < -EPS) | (comparison["delta_r_V5_minus_V4"] > 0)),
    ],
    ["V5 dominates", "V4 dominates"],
    default="tradeoff/tie",
)

comparison_cols = [
    "case",
    "W_V4", "K_V4", "P_uniform_V4",
    "P_valid_V4", "P_valid_V5", "delta_P_V5_minus_V4", "P_winner",
    "r_opt_V4", "r_opt_V5", "delta_r_V5_minus_V4", "fewer_repetitions",
    "theta_over_pi_V4", "theta_over_pi_V5",
    "beta_over_pi_V4", "beta_over_pi_V5",
    "P_valid_per_repetition_V4", "P_valid_per_repetition_V5", "efficiency_winner",
    "estimated_total_qubits_V4", "estimated_total_qubits_V5", "delta_qubits_V5_minus_V4",
    "pareto_probability_repetitions",
]
comparison = comparison[comparison_cols]
comparison.to_csv(OUT_DIR / "v4_v5_case_comparison.csv", index=False)

print_table(comparison, "V4 vs V5 by case")



V4 vs V5 by case
                            case  W_V4  K_V4  P_uniform_V4  P_valid_V4  P_valid_V5  delta_P_V5_minus_V4 P_winner  r_opt_V4  r_opt_V5  delta_r_V5_minus_V4 fewer_repetitions  theta_over_pi_V4  theta_over_pi_V5  beta_over_pi_V4  beta_over_pi_V5  P_valid_per_repetition_V4  P_valid_per_repetition_V5 efficiency_winner  estimated_total_qubits_V4  estimated_total_qubits_V5  delta_qubits_V5_minus_V4 pareto_probability_repetitions
           01_1d_tiny_single_gap     5     1      0.200000    0.763423    0.891319             0.127896       V5        29        39                   10                V4          0.089286          0.678571         0.053571         0.459821                   0.026325                   0.022854                V4                         11                         18                         7                   tradeoff/tie
            02_1d_main_reference     7     2      0.285714    0.927011    0.868942            -0.058069       V4        32        36

In [6]:
# =========================================================
# Aggregate scorecard
# =========================================================

def count_winners(series):
    return {
        "V4": int((series == "V4").sum()),
        "V5": int((series == "V5").sum()),
        "tie": int((series == "tie").sum()),
    }


scorecard = {
    "cases": int(len(comparison)),
    "V4_mean_P_valid": float(comparison["P_valid_V4"].mean()),
    "V5_mean_P_valid": float(comparison["P_valid_V5"].mean()),
    "mean_delta_P_V5_minus_V4": float(comparison["delta_P_V5_minus_V4"].mean()),
    "V4_median_P_valid": float(comparison["P_valid_V4"].median()),
    "V5_median_P_valid": float(comparison["P_valid_V5"].median()),
    "V4_mean_r_opt": float(comparison["r_opt_V4"].mean()),
    "V5_mean_r_opt": float(comparison["r_opt_V5"].mean()),
    "mean_delta_r_V5_minus_V4": float(comparison["delta_r_V5_minus_V4"].mean()),
    "V4_mean_efficiency_P_per_r": float(comparison["P_valid_per_repetition_V4"].mean()),
    "V5_mean_efficiency_P_per_r": float(comparison["P_valid_per_repetition_V5"].mean()),
    "V4_mean_estimated_qubits": float(comparison["estimated_total_qubits_V4"].mean()),
    "V5_mean_estimated_qubits": float(comparison["estimated_total_qubits_V5"].mean()),
    "mean_delta_qubits_V5_minus_V4": float(comparison["delta_qubits_V5_minus_V4"].mean()),
    "P_valid_wins": count_winners(comparison["P_winner"]),
    "fewer_repetition_wins": count_winners(comparison["fewer_repetitions"]),
    "efficiency_wins": count_winners(comparison["efficiency_winner"]),
    "pareto_counts": comparison["pareto_probability_repetitions"].value_counts().to_dict(),
}
scorecard_df = pd.DataFrame([scorecard])
scorecard_df.to_csv(OUT_DIR / "v4_v5_scorecard.csv", index=False)
print("Scorecard:")
for key, value in scorecard.items():
    print(f"  {key}: {value}")

summary_rows = []
for version, df in [("V4", v4_results), ("V5", v5_results)]:
    summary_rows.append({
        "version": version,
        "oracle": VERSION_SPECS[version]["oracle_label"],
        "mean_P_valid": float(df["P_valid"].mean()),
        "median_P_valid": float(df["P_valid"].median()),
        "min_P_valid": float(df["P_valid"].min()),
        "max_P_valid": float(df["P_valid"].max()),
        "mean_r_opt": float(df["r_opt"].mean()),
        "median_r_opt": float(df["r_opt"].median()),
        "mean_improvement_over_uniform": float(df["improvement_over_uniform"].mean()),
        "mean_P_valid_per_repetition": float(df["P_valid_per_repetition"].mean()),
        "mean_estimated_total_qubits": float(df["estimated_total_qubits"].mean()),
    })
version_summary = pd.DataFrame(summary_rows)
version_summary.to_csv(OUT_DIR / "v4_v5_version_summary.csv", index=False)
print_table(version_summary, "Version-level summary")


Scorecard:
  cases: 10
  V4_mean_P_valid: 0.9080353952999174
  V5_mean_P_valid: 0.8713485766594047
  mean_delta_P_V5_minus_V4: -0.03668681864051258
  V4_median_P_valid: 0.9301198136993125
  V5_median_P_valid: 0.8801303530594359
  V4_mean_r_opt: 30.6
  V5_mean_r_opt: 32.0
  mean_delta_r_V5_minus_V4: 1.4
  V4_mean_efficiency_P_per_r: 0.040914485213914505
  V5_mean_efficiency_P_per_r: 0.04049262495904205
  V4_mean_estimated_qubits: 28.4
  V5_mean_estimated_qubits: 38.8
  mean_delta_qubits_V5_minus_V4: 10.4
  P_valid_wins: {'V4': 7, 'V5': 3, 'tie': 0}
  fewer_repetition_wins: {'V4': 5, 'V5': 3, 'tie': 2}
  efficiency_wins: {'V4': 7, 'V5': 3, 'tie': 0}
  pareto_counts: {'V4 dominates': 6, 'tradeoff/tie': 2, 'V5 dominates': 2}

Version-level summary
version                 oracle  mean_P_valid  median_P_valid  min_P_valid  max_P_valid  mean_r_opt  median_r_opt  mean_improvement_over_uniform  mean_P_valid_per_repetition  mean_estimated_total_qubits
     V4        C(i) cost phase      0.908035

In [7]:
# =========================================================
# Figures: probability, repetitions, efficiency, and qubits
# =========================================================

x = np.arange(len(comparison))

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - 0.25, comparison["P_uniform_V4"], width=0.20, color="0.75", label="uniform K/W")
ax.bar(x, comparison["P_valid_V4"], width=0.24, color=V4_COLOR, label="V4 C(i)")
ax.bar(x + 0.25, comparison["P_valid_V5"], width=0.24, color=V5_COLOR, label="V5 ΔC(i)")
for xi, row in comparison.iterrows():
    winner = row["P_winner"]
    label = winner if winner != "tie" else "="
    y = max(row["P_valid_V4"], row["P_valid_V5"]) + 0.025
    ax.text(xi, min(1.06, y), label, ha="center", va="bottom", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(case_labels, rotation=45, ha="right")
ax.set_ylabel("P_valid")
ax.set_ylim(0, 1.12)
ax.set_title("V4 vs V5: best optimized P_valid by case")
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.18))
fig.tight_layout()
savefig(fig, "v4_v5_case_probabilities")

fig, ax = plt.subplots(figsize=(12, 5))
delta = comparison["delta_P_V5_minus_V4"].to_numpy()
colors = [POS_COLOR if d > EPS else NEG_COLOR if d < -EPS else TIE_COLOR for d in delta]
ax.bar(x, delta, color=colors, alpha=0.9)
ax.axhline(0, color=BASELINE_COLOR, linestyle="--", linewidth=1.2)
for xi, d in zip(x, delta):
    va = "bottom" if d >= 0 else "top"
    offset = 0.015 if d >= 0 else -0.015
    ax.text(xi, d + offset, f"{d:+.3f}", ha="center", va=va, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(case_labels, rotation=45, ha="right")
ax.set_ylabel("ΔP_valid = V5 - V4")
ax.set_title("Probability advantage of V5 over V4")
fig.tight_layout()
savefig(fig, "v4_v5_delta_probability_by_case")

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - 0.18, comparison["r_opt_V4"], width=0.34, color=V4_COLOR, label="V4")
ax.bar(x + 0.18, comparison["r_opt_V5"], width=0.34, color=V5_COLOR, label="V5")
for xi, row in comparison.iterrows():
    label = row["fewer_repetitions"] if row["fewer_repetitions"] != "tie" else "="
    y = max(row["r_opt_V4"], row["r_opt_V5"]) + 1
    ax.text(xi, y, label, ha="center", va="bottom", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(case_labels, rotation=45, ha="right")
ax.set_ylabel("optimal repetitions r_opt")
ax.set_title("V4 vs V5: repetitions needed by selected optimum")
ax.legend()
fig.tight_layout()
savefig(fig, "v4_v5_repetitions_by_case")

fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(comparison["r_opt_V4"], comparison["P_valid_V4"], s=75, color=V4_COLOR, label="V4")
ax.scatter(comparison["r_opt_V5"], comparison["P_valid_V5"], s=75, color=V5_COLOR, label="V5")
for _, row in comparison.iterrows():
    short = row["case"].split("_", 1)[0]
    ax.annotate(short, (row["r_opt_V4"], row["P_valid_V4"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
    ax.annotate(short, (row["r_opt_V5"], row["P_valid_V5"]), xytext=(4, -10), textcoords="offset points", fontsize=8)
ax.set_xlabel("r_opt")
ax.set_ylabel("P_valid")
ax.set_ylim(0, 1.05)
ax.set_title("Tradeoff: probability vs repetitions")
ax.legend()
fig.tight_layout()
savefig(fig, "v4_v5_probability_vs_repetitions")

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - 0.18, comparison["P_valid_per_repetition_V4"], width=0.34, color=V4_COLOR, label="V4")
ax.bar(x + 0.18, comparison["P_valid_per_repetition_V5"], width=0.34, color=V5_COLOR, label="V5")
ax.set_xticks(x)
ax.set_xticklabels(case_labels, rotation=45, ha="right")
ax.set_ylabel("P_valid / r_opt")
ax.set_title("Simple efficiency score: P_valid per repetition")
ax.legend()
fig.tight_layout()
savefig(fig, "v4_v5_efficiency_by_case")

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - 0.18, comparison["estimated_total_qubits_V4"], width=0.34, color=V4_COLOR, label="V4")
ax.bar(x + 0.18, comparison["estimated_total_qubits_V5"], width=0.34, color=V5_COLOR, label="V5")
ax.set_xticks(x)
ax.set_xticklabels(case_labels, rotation=45, ha="right")
ax.set_ylabel("estimated total qubits")
ax.set_title("Register-size cost of V5 extra m2 and ancilla registers")
ax.legend()
fig.tight_layout()
savefig(fig, "v4_v5_estimated_qubits_by_case")

print(f"Figures saved in: {OUT_DIR}")


Figures saved in: analysis_v4_v5_comparison


In [8]:
# =========================================================
# Repetition curves for the selected theta/beta of each version
# =========================================================

for case in case_order:
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    for version, color in [("V4", V4_COLOR), ("V5", V5_COLOR)]:
        part = all_curves[(all_curves["case"] == case) & (all_curves["version"] == version)].sort_values("r")
        if part.empty:
            continue
        ax.plot(part["r"], part["P_valid"], color=color, linewidth=2, label=version)
        selected = part[part["is_selected_r"] == 1]
        if not selected.empty:
            ax.scatter(selected["r"], selected["P_valid"], color=color, s=55, edgecolor="black", linewidth=0.5)
    p_uniform = float(comparison.loc[comparison["case"] == case, "P_uniform_V4"].iloc[0])
    ax.axhline(p_uniform, color=BASELINE_COLOR, linestyle="--", linewidth=1.2, label="uniform K/W")
    ax.set_xlabel("repetitions r")
    ax.set_ylabel("P_valid")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"Selected-angle repetition curve - {case}")
    ax.legend()
    fig.tight_layout()
    savefig(fig, f"{slug(case)}_selected_repetition_curve")

print(f"Per-case repetition curves saved in: {OUT_DIR}")


Per-case repetition curves saved in: analysis_v4_v5_comparison


In [9]:
# =========================================================
# Compact conclusion table
# =========================================================

conclusion_rows = []
for _, row in comparison.iterrows():
    conclusion_rows.append({
        "case": row["case"],
        "best_probability": row["P_winner"],
        "delta_P_V5_minus_V4": row["delta_P_V5_minus_V4"],
        "fewer_repetitions": row["fewer_repetitions"],
        "delta_r_V5_minus_V4": row["delta_r_V5_minus_V4"],
        "best_efficiency": row["efficiency_winner"],
        "pareto_probability_repetitions": row["pareto_probability_repetitions"],
    })
conclusion = pd.DataFrame(conclusion_rows)
conclusion.to_csv(OUT_DIR / "v4_v5_compact_conclusion.csv", index=False)
print_table(conclusion, "Compact per-case conclusion")

print("\nMain takeaways:")
print(f"- Mean P_valid: V4={scorecard['V4_mean_P_valid']:.4f}, V5={scorecard['V5_mean_P_valid']:.4f}, delta={scorecard['mean_delta_P_V5_minus_V4']:+.4f}.")
print(f"- Probability wins: {scorecard['P_valid_wins']}.")
print(f"- Fewer-repetition wins: {scorecard['fewer_repetition_wins']}.")
print(f"- Efficiency wins (P_valid/r): {scorecard['efficiency_wins']}.")
print(f"- Mean estimated qubits: V4={scorecard['V4_mean_estimated_qubits']:.1f}, V5={scorecard['V5_mean_estimated_qubits']:.1f}.")



Compact per-case conclusion
                            case best_probability  delta_P_V5_minus_V4 fewer_repetitions  delta_r_V5_minus_V4 best_efficiency pareto_probability_repetitions
           01_1d_tiny_single_gap               V5             0.127896                V4                   10              V4                   tradeoff/tie
            02_1d_main_reference               V4            -0.058069                V4                    4              V4                   V4 dominates
          03_1d_two_free_regions               V4            -0.021678                V4                   12              V4                   V4 dominates
          04_1d_clustered_medium               V4            -0.122138               tie                    0              V4                   V4 dominates
     05_1d_long_clustered_blocks               V4            -0.119288                V4                    8              V4                   V4 dominates
         06_2d_tiny_corner_bl